# 10 · Attack stage (MITRE ATT&CK)

How far did each attacker get? Four stages, in order: **recon** (looking around), **intrusion** (trying to get in), **exploitation** (it worked), **data_theft** (taking data). The rules are in `ml/stage.py`.

A log does not say whether an attack worked, so *worked* is read from the answer: a 2xx status with a response much larger than this endpoint normally sends. *Normally* is learned from the same log, so it works on any site.

**Honest note:** there is no ground truth for the stage. These are rules we wrote, and the Random Forest at the end only shows that a model can learn them (as the proposal planned). The next-step model (Markov chain) needs attackers that go through several stages, which the generator does not make yet, so it comes in a later notebook.

Development data only (run3, run4).

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from eval.final_test import run
from ml.stage import STAGES, add_stages

D = ROOT / 'data'
RUNS = {'Juice Shop (run3)': (D / 'raw' / 'juiceshop_generated_run3.log', D / 'labels' / 'requests_run3.csv'),
        'Juice Shop (run4)': (D / 'raw' / 'juiceshop_generated_run4.log', D / 'labels' / 'requests_run4.csv'),
        'DVWA (run4)': (D / 'raw' / 'dvwa_generated_run4.log', D / 'labels' / 'requests_dvwa_run4.csv')}
results = {}
for name, (log, labels) in RUNS.items():
    req, sessions, _ = run(log, labels)
    results[name] = add_stages(req, sessions)

## 1. Stage of each request, by true attack type

Normal requests should get no stage at all (`-`).

In [2]:
for name, (req, _) in results.items():
    print(name)
    display(pd.crosstab(req['attack_type'].replace('', 'normal'), req['stage'].replace('', '-'))
            .reindex(columns=['-'] + STAGES, fill_value=0))

Juice Shop (run3)


stage,-,recon,intrusion,exploitation,data_theft
attack_type,,,,,
bruteforce,0,0,93,0,0
cmdi,1,0,96,0,0
enum,0,90,0,0,349
normal,3328,0,0,0,0
scan,0,247,0,0,0
sqli,1,0,74,1,0
traversal,0,0,58,0,0
xss,0,0,59,0,0


Juice Shop (run4)


stage,-,recon,intrusion,exploitation,data_theft
attack_type,,,,,
bruteforce,0,0,81,0,0
cmdi,0,0,65,0,0
enum,0,6,0,0,348
normal,3269,0,0,0,0
scan,0,282,0,0,0
sqli,0,0,65,2,0
traversal,0,0,74,0,0
xss,0,0,102,2,0


DVWA (run4)


stage,-,recon,intrusion,exploitation,data_theft
attack_type,,,,,
bruteforce,0,0,82,0,0
cmdi,0,0,60,0,0
enum,0,0,0,0,298
normal,2588,0,0,0,0
scan,0,305,0,0,0
sqli,0,0,64,0,0
traversal,0,0,76,0,0
xss,0,0,94,0,0


## 2. Furthest stage per session, and the new `critical` level

A session is **critical** when it is already high risk and reached exploitation or data theft.

In [3]:
for name, (_, sessions) in results.items():
    print(name)
    display(pd.crosstab(sessions['truth'], sessions['stage'].replace('', '-')))
    display(pd.crosstab(sessions['truth'], sessions['risk_level']))

Juice Shop (run3)


stage,-,data_theft,exploitation,intrusion,recon
truth,,,,,
attacker,0,19,1,58,12
normal,210,0,0,0,0


risk_level,critical,high,medium,none
truth,,,,
attacker,5,52,33,0
normal,0,0,0,210


Juice Shop (run4)


stage,-,data_theft,exploitation,intrusion,recon
truth,,,,,
attacker,0,14,3,59,14
normal,210,0,0,0,0


risk_level,critical,high,medium,none
truth,,,,
attacker,1,54,35,0
normal,0,0,0,210


DVWA (run4)


stage,-,data_theft,intrusion,recon
truth,,,,
attacker,0,14,62,14
normal,210,0,0,0


risk_level,high,medium,none
truth,,,
attacker,23,67,0
normal,0,0,210


## 3. What counted as exploitation

Every request marked *exploitation*, to check by eye that the answer really was unusual.

In [4]:
cols = ['ip', 'method', 'path', 'status', 'size', 'attack_type']
pd.concat({name: req.loc[req['stage'] == 'exploitation', cols] for name, (req, _) in results.items()})

ip method  \
Juice Shop (run3) 729    10.1.0.50    GET   
Juice Shop (run4) 2293  10.1.0.158    GET   
                  2574  10.1.0.177    GET   
                  3741    10.1.1.7   POST   
                  3742    10.1.1.7   POST   

                                                                     path  \
Juice Shop (run3) 729           /rest/products/search?q=%27+OR+1%3D1%2500   
Juice Shop (run4) 2293  /rest/products/search?q=%27+oR+1%3D1+OR+%27%27...   
                  2574   /rest/products/search?q=%25%27+Or+%271%27%3D%271   
                  3741                                        /api/Users/   
                  3742                                        /api/Users/   

                        status  size attack_type  
Juice Shop (run3) 729      200  5123        sqli  
Juice Shop (run4) 2293     200  5064        sqli  
                  2574     200  5064        sqli  
                  3741     201   325         xss  
                  3742     201   339         xss

## 4. Can a Random Forest learn the stage?

Trained on run3, tested on run4 (both sites). The features are what the rules look at (status, how much larger the answer is than normal, the session behaviour, whether the model flagged it, whether it is a probe for a hidden file), not the stage itself. High accuracy here means the model reproduces our rules, nothing more.

In [5]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from ml.stage import _normal_sizes, probe_only

def stage_features(req):
    normal = _normal_sizes(req)
    ep = req['path'].str.split('?').str[0]
    base = np.array([normal.get((e, s), np.nan) for e, s in zip(ep, req['status'])], dtype=float)
    size = pd.to_numeric(req['size'], errors='coerce').fillna(0).values
    # No raw status or size: they differ between sites, and the rules do not use them directly.
    X = pd.DataFrame({'pred': req['pred'].values,
                      'ok': req['status'].between(200, 299).astype(int).values,
                      'no_baseline': np.isnan(base).astype(int),
                      'size_ratio': np.where(np.isnan(base), 0, size / np.maximum(base, 1)),
                      'probe': probe_only(req).astype(int).values})
    for b in ['scan', 'enumeration', 'bruteforce']:
        X['beh_' + b] = (req['behaviour'] == b).astype(int).values
    return X

train_req = results['Juice Shop (run3)'][0]
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=2)
rf.fit(stage_features(train_req), train_req['stage'].replace('', '-'))
for name in ['Juice Shop (run4)', 'DVWA (run4)']:
    req = results[name][0]
    y, p = req['stage'].replace('', '-'), rf.predict(stage_features(req))
    print(name, 'accuracy vs the rules:', round(accuracy_score(y, p), 4))
    print(classification_report(y, p, zero_division=0))

Juice Shop (run4) accuracy vs the rules: 0.9995
              precision    recall  f1-score   support

           -       1.00      1.00      1.00      3269
  data_theft       1.00      1.00      1.00       348
exploitation       1.00      0.50      0.67         4
   intrusion       0.99      1.00      1.00       387
       recon       1.00      1.00      1.00       288

    accuracy                           1.00      4296
   macro avg       1.00      0.90      0.93      4296
weighted avg       1.00      1.00      1.00      4296



DVWA (run4) accuracy vs the rules: 0.9319
              precision    recall  f1-score   support

           -       1.00      0.99      0.99      2588
  data_theft       0.89      1.00      0.94       298
   intrusion       0.72      0.78      0.75       376
       recon       0.69      0.60      0.64       305

    accuracy                           0.93      3567
   macro avg       0.82      0.84      0.83      3567
weighted avg       0.93      0.93      0.93      3567



## 5. Conclusions

- Normal requests get no stage, and no normal session is critical.
- Most attackers stop at **intrusion**: the generator's payloads rarely get an unusual answer. The few *exploitation* cases on Juice Shop are real (for example `' OR 1=1` returning the whole product list).
- Enumeration that gets objects back (200) counts as **data_theft**: someone read objects one by one.
- **Limitation:** DVWA answers 200 with a normal-sized page whether an attack worked or not, so on DVWA exploitation cannot be seen from the log.
- The Random Forest reproduces the rules on the same site (Juice Shop run4: 99.95%), which only confirms the stage is a rule-based definition. On DVWA it drops to about 93%: it learned Juice Shop habits that the rules do not use (for example, on Juice Shop a failed login always gets 401, on DVWA it gets 200, so the forest calls DVWA's brute force recon). **So the system uses the rules for the stage, and the forest is kept only as the comparison the proposal asked for.**